In [1]:
import pandas as pd

In [2]:
# 1. CARGA DE DATOS
# Cargamos los datos limpios que exportamos en el EDA
df_detalle = pd.read_csv(r'C:\Users\nata_\Documents\MLOps\Introduction_to_MLOps\proyecto_2\Combos_Tostao\data\datos_tostao_limpios.csv')

# Convertimos la fecha a tipo datetime para que mantenga sus propiedades organizadas
df_detalle['fecha'] = pd.to_datetime(df_detalle['fecha'])

print(f"¡Datos cargados correctamente en el nuevo Notebook! Total de registros: {len(df_detalle)}")
df_detalle.head()

¡Datos cargados correctamente en el nuevo Notebook! Total de registros: 18376


,id_ticket,id_producto,cantidad,precio_unitario,nombre,categoria,subcategoria,fecha,id_tienda,id_cliente,venta_total,mes,dia_semana
0,TICKET_000001,PROD_030,1,4500,Torta de Zanahoria,Alimentos,Panadería Dulce,2024-08-29,TOSTAO_15,CUST_01364,4500,August,Thursday
1,TICKET_000002,PROD_006,1,2500,Aromática,Bebidas,Calientes,2024-02-04,TOSTAO_18,CUST_01956,2500,February,Sunday
2,TICKET_000003,PROD_033,1,1500,Chocolatina,Alimentos,Panadería Dulce,2024-02-23,TOSTAO_18,CUST_01501,1500,February,Friday
3,TICKET_000003,PROD_021,1,3500,Pastel de Pollo,Alimentos,Panadería Sal,2024-02-23,TOSTAO_18,CUST_01501,3500,February,Friday
4,TICKET_000004,PROD_019,1,3200,Empanada de Carne,Alimentos,Panadería Sal,2024-10-30,TOSTAO_08,CUST_01276,3200,October,Wednesday


In [3]:

# 1. Aseguramos el orden cronológico absoluto de las transacciones
df_ordenado = df_detalle.sort_values(by='fecha').reset_index(drop=True)

# 2. Calculamos la posición exacta para el corte del 80%
punto_corte = int(len(df_ordenado) * 0.80)
fecha_corte = df_ordenado.iloc[punto_corte]['fecha']

# 3. Dividimos el conjunto de datos
df_train_raw = df_ordenado.iloc[:punto_corte].copy()
df_test_raw = df_ordenado.iloc[punto_corte:].copy()

# 4. Métricas para auditar la división
total_filas = len(df_ordenado)
filas_train = len(df_train_raw)
filas_test = len(df_test_raw)

print("==================================================================")
print("🔍 AUDITORÍA PASO 1: DIVISIÓN TEMPORAL CRONOLÓGICA")
print("==================================================================")
print(f"📦 Dataset Completo    : {total_filas:,} filas de transacciones")
print(f"📅 Fecha de Corte      : {fecha_corte}")
print("-" * 66)
print(f"🟢 Entrenamiento (Train): {filas_train:,} filas ({filas_train/total_filas:.1%})")
print(f"   • Rango de fechas   : Desde {df_train_raw['fecha'].min().strftime('%Y-%m-%d')} hasta {df_train_raw['fecha'].max().strftime('%Y-%m-%d')}")
print("-" * 66)
print(f"🔵 Validación (Test)   : {filas_test:,} filas ({filas_test/total_filas:.1%})")
print(f"   • Rango de fechas   : Desde {df_test_raw['fecha'].min().strftime('%Y-%m-%d')} hasta {df_test_raw['fecha'].max().strftime('%Y-%m-%d')}")
print("==================================================================")

🔍 AUDITORÍA PASO 1: DIVISIÓN TEMPORAL CRONOLÓGICA
📦 Dataset Completo    : 18,376 filas de transacciones
📅 Fecha de Corte      : 2024-10-17 00:00:00
------------------------------------------------------------------
🟢 Entrenamiento (Train): 14,700 filas (80.0%)
   • Rango de fechas   : Desde 2024-01-01 hasta 2024-10-17
------------------------------------------------------------------
🔵 Validación (Test)   : 3,676 filas (20.0%)
   • Rango de fechas   : Desde 2024-10-17 hasta 2024-12-30


In [4]:
# Preparación de Datos (One-Hot Encoding)

def aplicar_one_hot_encoding(df_insumo: pd.DataFrame) -> pd.DataFrame:
    """
    Transforma el formato transaccional a una estructura matricial
    donde las columnas son los productos y las celdas indican si el
    producto fue comprado (1) o no (0) en cada ticket.
    """
    # 1. Agrupamos por ticket y producto sumando cantidades
    matriz_pivot = df_insumo.groupby(['id_ticket', 'nombre'])['cantidad'].sum().unstack().reset_index().fillna(0)
    
    # 2. Establecemos el id_ticket como índice de la matriz
    matriz_pivot.set_index('id_ticket', inplace=True)
    
    # 3. Codificación binaria: Si la cantidad es mayor a 0 se asigna 1, si no 0
    matriz_binaria = matriz_pivot.map(lambda x: 1 if x > 0 else 0)
    
    return matriz_binaria

print("==================================================================")
print("⚙️  EJECUTANDO PASO 2: ONE-HOT ENCODING MULTI-DATASET")
print("==================================================================")

# Aplicamos la transformación de canasta a Train y Test por separado
basket_train = aplicar_one_hot_encoding(df_train_raw)
basket_test = aplicar_one_hot_encoding(df_test_raw)

print(f"🟢 Dimensiones de la matriz de Train : {basket_train.shape[0]} tickets x {basket_train.shape[1]} productos")
print(f"🔵 Dimensiones de la matriz de Test  : {basket_test.shape[0]} tickets x {basket_test.shape[1]} productos")
print("------------------------------------------------------------------")


⚙️  EJECUTANDO PASO 2: ONE-HOT ENCODING MULTI-DATASET
🟢 Dimensiones de la matriz de Train : 7991 tickets x 35 productos
🔵 Dimensiones de la matriz de Test  : 2010 tickets x 35 productos
------------------------------------------------------------------


In [5]:
import os

print("==================================================================")
print("EJECUTANDO PASO 3: EXPORTACIÓN DE MATRICES (REPRODUCIBILIDAD)")
print("==================================================================")

# Exportar las matrices binarias (One-Hot Encoded) a formato CSV
basket_train.to_csv('train_basket.csv')
basket_test.to_csv('test_basket.csv')

print("🟢 Archivo guardado de manera exitosa: 'train_basket.csv'")
print("🔵 Archivo guardado de manera exitosa: 'test_basket.csv'")
print("------------------------------------------------------------------")


EJECUTANDO PASO 3: EXPORTACIÓN DE MATRICES (REPRODUCIBILIDAD)
🟢 Archivo guardado de manera exitosa: 'train_basket.csv'
🔵 Archivo guardado de manera exitosa: 'test_basket.csv'
------------------------------------------------------------------


In [8]:
import sys
!{sys.executable} -m pip install mlflow

  Using cached mlflow-3.12.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.12.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-7.1.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.110.0-py3-none-any.whl.metadata (43 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t


   ------ --------------------------------- 4.5/27.5 MB 6.6 MB/s eta 0:00:04
   --------- ------------------------------ 6.8/27.5 MB 7.2 MB/s eta 0:00:03
   ------------ --------------------------- 8.7/27.5 MB 7.4 MB/s eta 0:00:03
   ---------------- ----------------------- 11.3/27.5 MB 7.4 MB/s eta 0:00:03
   ------------------ --------------------- 12.6/27.5 MB 7.0 MB/s eta 0:00:03
   -------------------- ------------------- 14.4/27.5 MB 7.0 MB/s eta 0:00:02
   ----------------------- ---------------- 16.0/27.5 MB 6.9 MB/s eta 0:00:02
   --------------------------- ------------ 18.6/27.5 MB 7.1 MB/s eta 0:00:02
   ----------------------------- ---------- 20.2/27.5 MB 7.1 MB/s eta 0:00:02
   -------------------------------- ------- 22.0/27.5 MB 7.2 MB/s eta 0:00:01
   ---------------------------------- ----- 23.9/27.5 MB 7.3 MB/s eta 0:00:01
   ------------------------------------ --- 24.9/27.5 MB 7.3 MB/s eta 0:00:01
   ------------------------------------ --- 25.4/27.5 MB 6.9 MB/s 

In [10]:
import sys
!{sys.executable} -m pip install mlxtend

In [12]:
import mlflow

print("==================================================================")
print("CONFIGURACIÓN DE MLFLOW")
print("==================================================================")

# Definimos la ruta absoluta directa a la raíz del proyecto (fuera de notebooks)
MLRUNS_DIR = r"C:\Users\nata_\Documents\MLOps\Introduction_to_MLOps\proyecto_2\Combos_Tostao\mlruns"

# Configuración de URI y Experimento
mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR}")
mlflow.set_experiment("Market_Basket_Tostao")

print("------------------------------------------------------------------")
print("✅ ¡Entorno de MLflow inicializado correctamente!")
print(f"📁 Tracking de experimentos activo en: {MLRUNS_DIR}")
print("==================================================================")

c:\Users\nata_\anaconda3\envs\my_env\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/21 16:04:07 INFO mlflow.tracking.fluent: Experiment with name 'Market_Basket_Tostao' does not exist. Creating a new experiment.


CONFIGURACIÓN DE MLFLOW
------------------------------------------------------------------
✅ ¡Entorno de MLflow inicializado correctamente!
📁 Tracking de experimentos activo en: C:\Users\nata_\Documents\MLOps\Introduction_to_MLOps\proyecto_2\Combos_Tostao\mlruns


In [13]:
from mlxtend.frequent_patterns import apriori, association_rules

with mlflow.start_run(run_name="Soporte_1%"):
    # 1. Parámetros
    mlflow.log_param("min_support", 0.01)
    mlflow.log_param("min_confidence", 0.10)
    
    # 2. Algoritmo
    frequent_train = apriori(basket_train, min_support=0.01, use_colnames=True)
    reglas_train = association_rules(frequent_train, metric="confidence", min_threshold=0.1)
    
    # 3. Métricas
    total_reglas_crudas = len(reglas_train)
    lift_maximo = reglas_train['lift'].max() if not reglas_train.empty else 0
    
    mlflow.log_metric("reglas_descubiertas_crudas", total_reglas_crudas)
    mlflow.log_metric("lift_maximo_train", lift_maximo)
    
    print("==================================================================")
    print("🚀 EXPERIMENTO 1 GUARDADO EN MLFLOW")
    print("==================================================================")
    print(f"📊 Parámetros: Soporte = 1%, Confianza = 10%")
    print(f"📈 Métricas Guardadas: {total_reglas_crudas} reglas crudas encontradas.")

🚀 EXPERIMENTO 1 GUARDADO EN MLFLOW
📊 Parámetros: Soporte = 1%, Confianza = 10%
📈 Métricas Guardadas: 18 reglas crudas encontradas.


c:\Users\nata_\anaconda3\envs\my_env\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [14]:
from mlxtend.frequent_patterns import apriori, association_rules

with mlflow.start_run(run_name="Soporte_Relajado_0.5%"):
    # 1. Parámetros
    mlflow.log_param("min_support", 0.005)
    mlflow.log_param("min_confidence", 0.10)
    
    # 2. Algoritmo
    frequent_train_exp = apriori(basket_train, min_support=0.005, use_colnames=True)
    reglas_train_exp = association_rules(frequent_train_exp, metric="confidence", min_threshold=0.1)
    
    # 3. Métricas
    total_reglas_exp = len(reglas_train_exp)
    lift_maximo_exp = reglas_train_exp['lift'].max() if not reglas_train_exp.empty else 0
    
    mlflow.log_metric("reglas_descubiertas_crudas", total_reglas_exp)
    mlflow.log_metric("lift_maximo_train", lift_maximo_exp)
    
    print("==================================================================")
    print("🧪 EXPERIMENTO 2 GUARDADO EN MLFLOW")
    print("==================================================================")
    print(f"📊 Parámetros: Soporte = 0.5%, Confianza = 10%")
    print(f"📈 Métricas Guardadas: {total_reglas_exp} reglas crudas encontradas.")

🧪 EXPERIMENTO 2 GUARDADO EN MLFLOW
📊 Parámetros: Soporte = 0.5%, Confianza = 10%
📈 Métricas Guardadas: 19 reglas crudas encontradas.


c:\Users\nata_\anaconda3\envs\my_env\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [15]:
from mlxtend.frequent_patterns import apriori, association_rules

with mlflow.start_run(run_name="Soporte_Ultra_Bajo_0.2%"):
    # 1. Registramos parámetros en MLflow
    mlflow.log_param("min_support", 0.002)     # Un umbral muy bajo
    mlflow.log_param("min_confidence", 0.10)   # Mantenemos la confianza para comparar
    
    # 2. Ejecutamos el algoritmo
    frequent_train_exp3 = apriori(basket_train, min_support=0.002, use_colnames=True)
    reglas_train_exp3 = association_rules(frequent_train_exp3, metric="confidence", min_threshold=0.1)
    
    # 3. Métricas para el análisis de sensibilidad
    total_reglas_exp3 = len(reglas_train_exp3)
    lift_maximo_exp3 = reglas_train_exp3['lift'].max() if not reglas_train_exp3.empty else 0
    
    mlflow.log_metric("reglas_descubiertas_crudas", total_reglas_exp3)
    mlflow.log_metric("lift_maximo_train", lift_maximo_exp3)
    
    print("==================================================================")
    print("🔬 EXPERIMENTO 3 GUARDADO EN MLFLOW")
    print("==================================================================")
    print(f"📊 Parámetros: Soporte = 0.2%, Confianza = 10%")
    print(f"📈 Métricas Guardadas: {total_reglas_exp3} reglas crudas encontradas.")

c:\Users\nata_\anaconda3\envs\my_env\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


🔬 EXPERIMENTO 3 GUARDADO EN MLFLOW
📊 Parámetros: Soporte = 0.2%, Confianza = 10%
📈 Métricas Guardadas: 52 reglas crudas encontradas.


In [16]:
from mlxtend.frequent_patterns import apriori, association_rules

with mlflow.start_run(run_name="Soporte_Extremo_0.1%"):
    # 1. Registramos el parámetro en MLflow
    mlflow.log_param("min_support", 0.001)     # El límite inferior absoluto
    mlflow.log_param("min_confidence", 0.10)   # Mantenemos la confianza fija
    
    # 2. Ejecutamos el algoritmo
    frequent_train_exp4 = apriori(basket_train, min_support=0.001, use_colnames=True)
    reglas_train_exp4 = association_rules(frequent_train_exp4, metric="confidence", min_threshold=0.1)
    
    # 3. Métricas
    total_reglas_exp4 = len(reglas_train_exp4)
    lift_maximo_exp4 = reglas_train_exp4['lift'].max() if not reglas_train_exp4.empty else 0
    
    mlflow.log_metric("reglas_descubiertas_crudas", total_reglas_exp4)
    mlflow.log_metric("lift_maximo_train", lift_maximo_exp4)
    
    print("==================================================================")
    print("💥 EXPERIMENTO 4 GUARDADO EN MLFLOW")
    print("==================================================================")
    print(f"📊 Parámetros: Soporte = 0.1%, Confianza = 10%")
    print(f"📈 Métricas Guardadas: {total_reglas_exp4} reglas crudas encontradas.")

c:\Users\nata_\anaconda3\envs\my_env\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


💥 EXPERIMENTO 4 GUARDADO EN MLFLOW
📊 Parámetros: Soporte = 0.1%, Confianza = 10%
📈 Métricas Guardadas: 257 reglas crudas encontradas.


In [18]:

# 1. Crear el diccionario con los datos limpios de los 4 experimentos
datos_experimentos = {
    "Soporte Mínimo": ["0.1%", "0.2%", "0.5%", "1.0%"],
    "Hiperparámetro (min_support)": [0.001, 0.002, 0.005, 0.010],
    "Reglas Descubiertas": [257, 52, 19, 18],
    "Lift Máximo": [13.770, 10.140, 8.293, 8.293],

}

# 2. Convertir a DataFrame de Pandas
df_resumen = pd.DataFrame(datos_experimentos)

# 3. Mostrar la tabla sencilla en el notebook
df_resumen

,Soporte Mínimo,Hiperparámetro (min_support),Reglas Descubiertas,Lift Máximo
0,0.1%,0.001,257,13.770
1,0.2%,0.002,52,10.140
2,0.5%,0.005,19,8.293
3,1.0%,0.010,18,8.293
